In [1]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference, time_inference_torch

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
import time
import os
import gc

In [2]:
# Constants
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
base_dir = "../../../Datasets/csv/UAVATTACK/"
real_data_path = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"
MODEL_NAME = "Stacking Classifier(LogisticRegression RandomForest SVC)"

models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

In [3]:
# Load the real dataset
try:
    real_df = pd.read_csv(real_data_path)
    real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

if real_df is None or 'label' not in real_df.columns:
    raise RuntimeError("Real data not loaded or missing 'label' column.")

Loaded real data


In [4]:
# Drop unwanted columns if present
drop_cols = [col for col in ['label', 'timestamp'] if col in real_df.columns]
X_real = real_df.drop(columns=drop_cols)
y_real = real_df['label']

# Encode labels if necessary
if y_real.dtype == object or y_real.dtype.name == 'category':
    le_real = LabelEncoder()
    y_real = le_real.fit_transform(y_real)

X_real = X_real.astype(np.float32)

# Split real data into test set only (since we're always training on synthetic)
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X_real, y_real, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_real
)

del real_df, X_train_real, y_train_real
gc.collect()

117

In [5]:
# Define base models
base_learners = [
    ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1200, solver='saga')),
    ('rf', RandomForestClassifier(random_state=RANDOM_STATE)),
    ('svc', SVC(probability=True, random_state=RANDOM_STATE))
]

# Meta model
meta_learner = LogisticRegression(random_state=RANDOM_STATE, max_iter=1200, solver='saga')

In [6]:
results = []

In [ ]:
for model in models_list:
    for n in sample_sizes:
        file_name = f"spoofing_{model}_{n}.csv"
        file_path = os.path.join(base_dir, file_name)
        try:
            df = pd.read_csv(file_path)
            print(f"\nProcessing {file_name}")
        except FileNotFoundError:
            print(f"Error: The file '{file_path}' was not found.")
            continue
        except Exception as e:
            print(f"An error occurred while loading {file_name}: {e}")
            continue

        if 'label' not in df.columns:
            print(f"Warning: 'label' column not found in {file_name}. Skipping.")
            del df
            gc.collect()
            continue

        X_synth = df.drop('label', axis=1)
        y_synth = df['label']

        scaler = StandardScaler()
        X_synth_scaled = scaler.fit_transform(X_synth)
        X_test_real_scaled = scaler.transform(X_test_real)

        stacking_model = StackingClassifier(
            estimators=base_learners,
            final_estimator=meta_learner,
            passthrough=False,
            n_jobs=-1
        )

        start_time = time.time()
        stacking_model.fit(X_synth_scaled, y_synth)
        train_time = time.time() - start_time

        # Evaluate on real test data
        y_pred = stacking_model.predict(X_test_real_scaled)
        accuracy = accuracy_score(y_test_real, y_pred)
        f1 = f1_score(y_test_real, y_pred, average='weighted', zero_division=0)
        print(f"Training on synthetic: {file_name}, testing on real data")
        print(classification_report(
            y_test_real,
            y_pred,
            zero_division=0
        ))
        print(f"F1 Score: {f1:.4f}")
        print(f"Accuracy: {accuracy:.4f}")

        # Model parameters
        params = stacking_model.get_params() if hasattr(stacking_model, "get_params") else {}
        # StackingClassifier does not have 'epochs', so set as "N/A"
        num_epochs = "N/A"

        _row = evaluate(
            y_test_real,
            y_pred,
            model_name=MODEL_NAME,
            dataset_name=file_name,
            family="Synthetic-Stacking",
            labels=list(range(2)),
            class_names=["benign", "malicious"],
            y_proba=stacking_model.predict_proba(X_test_real_scaled) if hasattr(stacking_model, "predict_proba") else None,
            y_train=y_synth,
            train_time_sec=train_time,
            num_epochs=num_epochs,
            parameters=str(params),
            seed=RANDOM_STATE,
            n_features=X_test_real_scaled.shape[1],
            n_train=len(y_synth),
            test_set="real_heldout_20pct",
            timing=time_inference(stacking_model.predict, X_test_real_scaled),
        )
        print(summary(_row))
        results.append(_row)

        del df, X_synth, y_synth, X_synth_scaled, X_test_real_scaled, y_pred
        gc.collect()

In [ ]:
# Save results to CSV
append_row(results, RESULTS_CSV)
print(f"Results appended to {RESULTS_CSV}")